# Basketball Team Classification — Exploration

Evaluates team classification approaches on three basketball video clips. Video clips and ground truth labels were sourced independently (sample data was not provided). A custom data pipeline was built covering YouTube clip extraction, YOLO-based player detection, and an interactive manual labeling tool in `notebooks/data_collection.ipynb`.

| Clip | Match-up | Difficulty |
|------|----------|------------|
| `clip1_easy.mp4` | Celtics vs Heat | Easy — white/green vs black/red |
| `clip2_hard.mp4` | Spurs vs Grizzlies | Hard — light blue vs dark navy |
| `clip3_edge.mp4` | Cavs vs Knicks Christmas | Edge — navy throwback vs white/orange |

**Accuracy is measured on valid single-player detections only.** Merged bounding boxes and ambiguous detections were marked `valid=False` during labeling and are excluded from all calculations.

**Before running:** upload the three `.mp4` files to `/content/`.

## Section 1 — Setup

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
# Florence-2 requires transformers==4.44.2 (tokenizer compat).
# Qwen2-VL requires transformers>=4.49 (SlidingWindowCache).
# They CANNOT coexist — pick which model to run by setting RUN_MODEL below,
# then restart runtime and run all cells from the top.
#
# Run 1: RUN_MODEL = "florence"  → gets Florence-2 results
# Run 2: RUN_MODEL = "qwen"     → gets Qwen2-VL results
# The other model's cell will be skipped automatically.

RUN_MODEL = "qwen"  # ← Change to "florence" or "qwen", then restart runtime

if RUN_MODEL == "florence":
    !pip install -q ultralytics opencv-python-headless transformers==4.44.2 accelerate torch torchvision
elif RUN_MODEL == "qwen":
    !pip install -q ultralytics opencv-python-headless transformers==4.49.0 accelerate torch torchvision qwen-vl-utils
else:
    !pip install -q ultralytics opencv-python-headless transformers accelerate torch torchvision

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ManuPrabakaran/vlm_team_classifier"
REPO_DIR = "/content/vlm_team_classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo ready at {REPO_DIR}")

In [ ]:
import cv2
import json
import numpy as np
from pathlib import Path
from ultralytics import YOLO

from src.utils import extract_frames, detect_players
from src.config import YOLO_CONFIDENCE

In [ ]:
yolo_model = YOLO("yolov8n.pt")

In [ ]:
clips = {
    "clip1_easy": {
        "video_path": "/content/clip1_easy.mp4",
        "gt_path": f"{REPO_DIR}/data/clip1_easy_ground_truth.json",
        "description": "Celtics vs Heat - contrasting jerseys (black/green vs black/red)",
        "remove_first_frame": True,
        "team_names": ("Celtics", "Heat"),
        # Celtics wearing statement (black) jersey — prior knowledge unreliable
        # Heat wearing standard — prior knowledge reliable
        # Either team special → neutral output labels for both
        "special_jersey": {"team0": True, "team1": False},
    },
    "clip2_hard": {
        "video_path": "/content/clip2_hard.mp4",
        "gt_path": f"{REPO_DIR}/data/clip2_hard_ground_truth.json",
        "description": "Spurs vs Grizzlies - similar colors (light blue vs dark navy)",
        "remove_first_frame": True,
        "team_names": ("Spurs", "Grizzlies"),
        "special_jersey": {"team0": False, "team1": False},  # standard jerseys
    },
    "clip3_edge": {
        "video_path": "/content/clip3_edge.mp4",
        "gt_path": f"{REPO_DIR}/data/clip3_edge_ground_truth.json",
        "description": "Cavs vs Knicks Christmas - alternate jerseys (navy throwback vs white/orange)",
        "remove_first_frame": False,
        "team_names": ("Cavaliers", "Knicks"),
        "special_jersey": {"team0": True, "team1": True},   # both wearing Christmas jerseys
    },
}

In [ ]:
# extract_frames returns [(frame, timestamp), ...] — strip timestamps for indexing
frames_dict = {}      # clip_name -> list of BGR numpy arrays
detections_dict = {}  # clip_name -> list of bbox lists (one per frame)
gt_dict = {}          # clip_name -> ground truth list

for clip_name, clip_info in clips.items():
    print(f"Processing {clip_name}...")

    # Extract frames at 1 FPS and strip timestamps
    frames = [f for f, _ in extract_frames(clip_info["video_path"], sample_rate=1)]

    # Run YOLO detection on each frame
    detections = [detect_players(f, yolo_model, YOLO_CONFIDENCE) for f in frames]

    # Remove the 0th extracted frame for clip1 and clip2.
    # The very first second is a pre-game shot; after removal, GT frame_idx 0
    # aligns with the next extracted frame.
    if clip_info["remove_first_frame"] and len(frames) > 1:
        frames.pop(0)
        detections.pop(0)

    frames_dict[clip_name] = frames
    detections_dict[clip_name] = detections

    with open(clip_info["gt_path"]) as f:
        gt_dict[clip_name] = json.load(f)

    print(f"  {len(frames)} video frames, {len(gt_dict[clip_name])} GT frames")

print("\nAll clips loaded.")

## Section 2 — K-Means Baseline

The baseline fits K-Means (k=2) on mean RGB jersey color from the first labeled frame, then predicts team for all subsequent detections. Label orientation is resolved by trying both 0/1 assignments and keeping the higher accuracy.

| Clip | Accuracy | Valid detections | Stability |
|------|----------|-----------------|----------|
| clip1_easy | 89.1% (82.6% in ~10% of runs) | 46 | Mostly stable — rare bad init |
| clip2_hard | 54.0% (stable) | 50 | Stable failure — colors too similar |
| clip3_edge | 90.6% or 81.2% (~50/50) | 32 | Bimodal — alternates almost every other run |

**Non-determinism**: `KMeans` has no `random_state`, producing three distinct regimes:

- **clip1** is mostly stable (89.1%) because the colors are distinct enough that initialization rarely matters — but drops to 82.6% in ~10% of runs, showing even easy cases aren't immune.
- **clip2** is stable at 54% for the opposite reason: colors are so similar that K-Means consistently fails regardless of initialization. No good local minimum exists.
- **clip3** alternates between exactly 90.6% and 81.2% on almost every other run — a bimodal distribution. K-Means has two equally-attractive local minima for this color distribution and random init decides which one it lands on.

**YOLO detection bias**: in clip2_hard, dark navy Grizzlies jerseys blend into the arena background, producing roughly a 4:1 ratio of Spurs to Grizzlies detections. This biases the K-Means centroids toward the over-represented team — detection quality and classification quality are coupled.

In [ ]:
from src.baseline import TeamClustering

In [ ]:
def evaluate_kmeans(frames, gt_data):
    """
    Fit K-Means on the first GT frame with >=2 valid non-referee player bboxes,
    then predict team for every valid non-referee detection across all GT frames.
    Tries both label orientations (0/1 swap) and returns the higher accuracy.

    Args:
        frames:  list of BGR numpy arrays indexed by GT frame_idx
        gt_data: list of dicts {frame_idx, timestamp, labels: [{bbox, team_id, valid}]}

    Returns:
        accuracy (float), n_detections (int)
    """
    # Find first frame with >=2 valid player bboxes (exclude referees from fit)
    fit_frame, fit_bboxes = None, []
    for entry in gt_data:
        player_bboxes = [
            label["bbox"]
            for label in entry["labels"]
            if label["valid"] and label["team_id"] != -1
        ]
        if len(player_bboxes) >= 2:
            fit_frame = frames[entry["frame_idx"]]
            fit_bboxes = player_bboxes
            break

    assert fit_frame is not None, (
        "No GT frame with >=2 valid player bboxes found. "
        "Check that frame indices align after first-frame removal."
    )

    tc = TeamClustering()
    tc.fit(fit_frame, fit_bboxes)

    predictions, ground_truth_labels = [], []
    for entry in gt_data:
        frame = frames[entry["frame_idx"]]
        for label in entry["labels"]:
            if not label["valid"] or label["team_id"] == -1:
                continue  # skip invalid boxes and referees
            predictions.append(int(tc.predict_team(frame, label["bbox"])))
            ground_truth_labels.append(label["team_id"])

    preds = np.array(predictions)
    gt    = np.array(ground_truth_labels)

    acc_normal  = float(np.mean(preds == gt))
    acc_flipped = float(np.mean((1 - preds) == gt))
    return max(acc_normal, acc_flipped), len(predictions)

In [ ]:
kmeans_results = {}

for clip_name, clip_info in clips.items():
    acc, n = evaluate_kmeans(frames_dict[clip_name], gt_dict[clip_name])
    kmeans_results[clip_name] = {
        "accuracy": round(acc, 3),
        "n_detections": n,
        "description": clip_info["description"],
    }
    print(f"{clip_name}: {acc:.1%}  ({n} detections)")

### Observations on K-Means Stability

Something interesting came up while running the baseline. The `TeamClustering` class initializes `KMeans` without a `random_state`, which means every run gets a different random starting point for the cluster centroids. In practice this turns out to matter a lot more than expected.

For **clip3_edge**, the accuracy flips between 90.6% and 81.2% on almost every other run. That is not just noise, it is the model landing on two completely different cluster solutions depending on where initialization starts. For **clip1_easy**, the result is usually 89.1% but falls to 82.6% about 10% of the time. Even with visually distinct jerseys, there is still a bad local minimum that the model occasionally gets stuck in.

The most paradoxical result is **clip2_hard**: it is actually the *most* stable across runs, locking in at 54.0% every time. That stability is not a good thing though. It just means the colors are so similar that K-Means finds the same wrong answer no matter where it starts.

This connects to color cluster separation. When colors are very distinct (clip1), initialization rarely changes the outcome. When colors are completely ambiguous (clip2), initialization also does not matter because there is no correct answer to find. clip3 sits right in the middle zone where starting centroids actually determine which of two valid-looking solutions the model converges to.

To get a more reliable picture, the cell below runs K-Means 10 times per clip and reports mean, std, min, and max accuracy.

In [ ]:
n_runs = 10
stability_results = {}

for clip_name in clips.keys():
    accs = []
    frames = frames_dict[clip_name]
    gt_data = gt_dict[clip_name]
    for _ in range(n_runs):
        acc, n = evaluate_kmeans(frames, gt_data)
        accs.append(acc)
    stability_results[clip_name] = {
        "mean": float(np.mean(accs)),
        "std": float(np.std(accs)),
        "min": float(np.min(accs)),
        "max": float(np.max(accs)),
        "n_detections": n,
    }
    print(
        f"{clip_name}: mean={np.mean(accs):.1%} +/- {np.std(accs):.1%} "
        f"(min={np.min(accs):.1%}, max={np.max(accs):.1%})"
    )

In [ ]:
metrics_path = Path(REPO_DIR) / "results" / "metrics.json"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
else:
    metrics = {}

metrics["baseline_kmeans"] = kmeans_results

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved to {metrics_path}")
print(json.dumps(metrics["baseline_kmeans"], indent=2))

## Section 2.5 — Automated Jersey Description Generation

The K-Means baseline has no concept of what a jersey looks like — it only sees mean RGB.
The VLM models in Section 3 need precise, model-aware descriptions to do better.
This section generates those descriptions automatically via GPT-4o, once per matchup.

### How we designed the prompts

**Start with individual descriptions.**
We wrote a structured prompt (v2) that asks GPT-4o to produce a flat list of machine-detectable
visual features for one jersey at a time — one feature per line, ordered from most visually
dominant to least. This ordering matters: Florence-2 front-weights early lines, so the most
diagnostic features must appear first. The v1 prompt produced unordered prose; v2 adds the
ordering constraint and removes a redundant no-comparison rule (the single-image framing
already prevents comparison by definition).

**Design descriptions to be contrastive, not just descriptive.**
A description that accurately describes one team's jersey is not necessarily useful for
classification. What matters is whether the description captures features that *differ* from
the other team. The prompt explicitly instructs GPT-4o to focus on distinctive and
high-contrast features — this is contrastive calibration done at description generation time,
not at inference time.

**Generate a differences description as an intermediate enrichment step.**
The differences prompt (`PROMPT_DIFFERENCES`) takes both jersey images simultaneously and
produces a concise list of what visually separates the two teams. This output is not used
directly by Florence-2 or Qwen2-VL at inference — it is fed as additional context INTO the
Qwen prompt generators (2B and 7B), so that when GPT-4o writes the final Qwen classification
prompt, it has explicit contrastive guidance rather than having to infer the key differences
from the individual descriptions alone. One extra API call that enriches the most important
generated output.

**Separate prompts for each target model.**
Florence-2 and Qwen2-VL have fundamentally different architectures and different prompt
capacities:

- **Florence-2** was designed for short task-token prompts (`<CAPTION>`, `<OD>`, etc.).
  Its 0.23B text decoder was not trained on long instruction-following inputs. Its prompt
  generator asks GPT-4o for 2-4 features per team — the highest-signal ones only, with shared
  features explicitly excluded (shared features waste attention budget without adding signal).

- **Qwen2-VL 2B** is instruction-tuned and can use richer context. Its generator asks for
  4-6 features per team, receives the differences description as additional input, and produces
  a structured but compact prompt. "Lost in the middle" is a real risk at 2B, so features are
  ordered by diagnostic strength.

- **Qwen2-VL 7B** can handle more structure. Its generator asks for 5-8 features per team
  and allows light formatting (bullet points). The larger model has less "lost in the middle"
  sensitivity, so the prompt can be slightly denser without losing fidelity.

**Label strategy driven by `special_jersey` flags.**
Standard jerseys use actual team names as classification labels — the model's prior knowledge
that "Celtics wear green and black" adds signal on top of the description. Special jerseys
(statement kits, Christmas editions, throwbacks) use neutral labels ("Team A", "Team B")
because the model's memorized appearance may be wrong, which would fight against the
description. The flag is tracked per team, not per clip — one team can be in a special jersey
while the other is not. If either team has a special jersey, both use neutral labels for
output consistency.

**Multiple images per team.**
Each team's folder can contain multiple images (front, back, shorts detail, side view). All
images in the folder are sent to GPT-4o together, giving the model a complete view of the
uniform before generating descriptions and prompts. This is especially useful for capturing
shorts features and back-of-jersey text that a single front-facing photo would miss.

In [ ]:
!pip install -q openai

from openai import OpenAI
import base64
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
print("OpenAI client ready.")

In [ ]:
import os

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def call_gpt4o(prompt_text, image_paths=None, model="gpt-4o", max_tokens=1000):
    """Call GPT-4o with a text prompt and optional list of image file paths."""
    content = []
    for p in (image_paths or []):
        ext = os.path.splitext(p)[1].lower().lstrip(".")
        mime = "jpeg" if ext in ("jpg", "jpeg") else ext
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/{mime};base64,{encode_image(p)}"}
        })
    content.append({"type": "text", "text": prompt_text})
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": content}],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print("GPT-4o helper defined.")

In [ ]:
# ── Prompt constants ─────────────────────────────────────────────────────────
# All prompts documented in Section 3 (Building Jersey Descriptions cell).
# PROMPT_INDIVIDUAL + PROMPT_DIFFERENCES: passed directly to GPT-4o with images.
# The three generator prompts produce ready-to-use classification prompts for
# each model — GPT-4o reads the generated descriptions + differences + images
# and outputs the final string passed to Florence-2 / Qwen at inference time.

PROMPT_INDIVIDUAL = """You are analyzing a single basketball uniform image.

Your output will be consumed by other AI models, including Florence-2, Qwen2-VL, and similar vision-language models. These models have limited attention and rely on short, high-signal, visually detectable features. You must write strictly for machine consumption.

---

TASK

Produce a list of distinctive, machine-detectable visual features that define the uniform.

---

REQUIREMENTS

- Use short, declarative sentences.
- Each sentence must describe exactly ONE visual feature.
- Order features from most visually dominant to least.
- Prefer features that are:
  - visually obvious
  - high-contrast
  - geometrically clear
  - consistently detectable by a model

Focus on:
- Dominant color(s) and color combinations
- Text presence and styling (color, outline, placement, size)
- Number styling (color, outline, placement), but NOT the specific number
- Trim and edge boundaries (neckline, arm openings)
- Stripes, bands, and geometric patterns
- Shorts features (waistband color, side stripes, bottom bands)

Strongly prefer:
- Features visible from multiple angles (front, back, side)
- Features that remain stable under partial occlusion

---

CRITICAL CONSTRAINTS

- Do NOT use the specific player name or number as a feature
- Do NOT rely on any identity that changes from player to player
- Only describe the STYLE and PLACEMENT of text and numbers

- Do NOT classify the team
- Do NOT include reasoning or explanations
- Do NOT include fabric, material, or stylistic commentary
- Avoid redundancy
- Avoid vague or subjective language

---

OUTPUT FORMAT

- A flat list of feature statements
- One sentence per line
- No grouping, no sections
- Order from most visually dominant feature to least

---

EXAMPLE OUTPUT

Black base uniform.
Green trim around neckline and arm openings.
White team name centered on chest.
Numbers have green fill with white outline.
Green waistband on shorts.
Thick green bands at the bottom of shorts.
Vertical green side stripes on shorts from waistband to hem.

---

Your goal is to produce a dense, consistent feature list that downstream AI systems can reliably use to recognize this uniform across different views."""

PROMPT_DIFFERENCES = """You are comparing two basketball uniforms side by side for visual recognition.

You are given two uniform images. Image 1 is Uniform A. Image 2 is Uniform B.

Your goal is to list the most visually distinctive differences between them — features that a downstream vision-language model can use to tell them apart in a real basketball game crop (chest to mid-thigh, arena lighting, motion blur possible).

---

RULES

- Each statement must describe ONE difference between the two uniforms.
- Use the format: "A has [feature]. B has [feature]." or "A is [quality] while B is [quality]."
- Order from most visually dominant difference to least.
- Focus only on differences that are high-contrast, visible from multiple angles, and likely to survive motion blur, glare, and partial crops.
- Do NOT describe features that are the same in both uniforms.
- Do NOT use player names or specific jersey numbers.
- Do NOT include reasoning, explanations, or subjective commentary.
- Use precise color names (e.g. "powder blue", "dark navy", "wine red").

---

OUTPUT FORMAT

- A flat list of difference statements
- One difference per line
- No grouping, no sections
- 4 to 8 differences maximum
- Ordered from most visually dominant to least

---

EXAMPLE OUTPUT

A has a black base. B has a white base.
A has green trim at neckline and arm openings. B has red trim.
A has green-filled numbers with white outline. B has red numbers with no outline.
A has thick green bands at bottom of shorts. B has no colored bands on shorts.
A has a green waistband. B has a white waistband.

---

Your goal is to produce a dense, ordered list of visual differences that a downstream AI model can use to reliably distinguish the two uniforms under real-game conditions."""

PROMPT_FLORENCE_GEN = """You are generating a final prompt for Florence-2.

Florence-2 is a prompt-based vision-language model that works best with short, concrete,
caption-like instructions and does not benefit much from long reasoning-heavy prompts.
The final prompt must therefore be extremely short and feature-first.

Inputs you will receive:
- Team 0 machine-readable uniform description
- Team 1 machine-readable uniform description
- optionally, reference images for Team 0 and Team 1

Task:
Create the shortest possible Florence-2 prompt that lets the model choose between Team 0
and Team 1 in a real basketball image.

Important environment constraints:
- real game footage may contain motion blur
- glare or bright reflections
- partial crops
- occlusion by other players
- fast movement

Because of this, prefer only large, high-contrast, easy-to-detect features. Ignore minor
or subtle details.

Feature selection rules:
- choose only 2 to 4 features per team
- prefer dominant colors, large text styling, major number styling, thick trim, bold
  stripes, large shorts bands, or major color blocking
- do not use player names
- do not use the specific jersey number
- only use text or number style, color, outline, and placement if visually distinctive
- discard any feature that is small, thin, low-contrast, or likely to disappear under
  blur or glare
- do not mention features shared by both teams — shared features add no signal and waste
  the model's attention
- if images are provided, use them to confirm or replace the weakest features in the
  descriptions; if not, use the descriptions alone

Prompt style rules:
- keep the final Florence-2 prompt extremely short
- put the strongest distinguishing feature first
- use simple concrete visual words
- do not include explanations
- do not include sections
- do not include reasoning steps
- use the labels 0 and 1 (not "Team A"/"Team B") — the response is parsed by code that
  looks for these exact characters
- end with: Output only 0 or 1.

Target form:
Use feature-to-label mapping, not abstract label discussion.

Example target style:
Black uniform with bright green trim and green shorts bands is 0.
White uniform with dark side panels and dark numbers is 1.
Which label matches this player crop? Output only 0 or 1.

Goal:
Produce the shortest Florence-2 prompt that preserves the strongest real-world visual
separation between the two teams."""

PROMPT_QWEN_2B_GEN = """You are generating a final classification prompt for Qwen2-VL 2B.

Qwen2-VL 2B is a small vision-language model. It can follow structured instructions but
loses attention in the middle of long prompts. It performs best when given compact,
high-signal, well-ordered information.

You will be given:
- Team 0 uniform feature list (machine-generated)
- Team 1 uniform feature list (machine-generated)
- optionally, a differences description comparing the two uniforms
- optionally, reference images for both teams

These feature lists already contain visually detectable properties.

---

TASK

Create a prompt that allows Qwen2-VL 2B to determine whether an observed uniform matches
Team 0, Team 1, or a Referee.

---

REAL-WORLD CONDITIONS

The input image may include:
- motion blur
- glare or reflections
- partial crops (torso only, back only, shorts only)
- occlusion by other players

Because of this:
- small or subtle features are unreliable
- large, high-contrast features are most important

---

FEATURE SELECTION RULE

From each team, select 4 to 6 features that:

- are visually dominant
- are high-contrast and easy to detect
- remain visible under blur, glare, and occlusion
- help distinguish the two teams from each other

Prioritize:
1) dominant color scheme
2) central text presence and styling
3) number styling (color and outline, NOT the specific number)
4) strong trim, bold stripes, or large color blocks
5) major shorts features (waistband color, large bands, side stripes)

Discard:
- small logos
- thin details
- low-contrast elements
- features shared by both teams

If a differences description is provided, include its key signals as a short final section
after both team descriptions. Order the differences by visual importance.

If reference images are provided, use them to confirm or replace weak features from the
descriptions.

---

LABEL RULES

Use the labels provided to you. Do not change them. The labels will be either:
- actual team names (e.g. "Spurs", "Grizzlies") when both teams wear standard jerseys
- neutral labels ("Team A", "Team B") when either team wears a special or alternate jersey

Always include "Referee" as a third output option, described as: grey or black and white
striped shirt, clearly different from both team uniforms.

For a non-special team in a mixed game (one standard, one special), include the team name
as context inside the description body even when the output label is neutral.

---

PROMPT STRUCTURE RULES

The prompt you create must:

- Present both team descriptions clearly and separately
- Use short, declarative feature phrases
- Order features from most diagnostic to least
- Place the most important features early
- Include a brief referee description
- End with a direct instruction to choose one label

---

CRITICAL CONSTRAINTS

- Do NOT use player names or specific numbers
- Only use style and placement of text and numbers
- Do NOT include long explanations
- Do NOT include chain-of-thought
- Do NOT include unnecessary words
- Keep total prompt length moderate and dense

---

OUTPUT FORMAT

The final prompt must end with exactly:
Output only [label0], [label1], or Referee.

where [label0] and [label1] are the labels provided to you.

---

EXAMPLE TARGET OUTPUT (neutral labels)

Decide which team this player belongs to, or identify them as a referee.

Team A: black base, bright green trim at neckline and arms, green waistband and thick green
bands at bottom of shorts, vertical green side stripes, white team name centered on chest,
green-filled numbers with white outline.

Team B: black base, red and white trim, white team name centered on chest, red and white
numbers, no green anywhere.

Key difference: Team A has green accents throughout. Team B has red and white accents.

Referee: grey or black and white striped shirt, clearly different from both teams.

Output only Team A, Team B, or Referee.

---

GOAL

Produce a compact, well-ordered prompt that maximizes correct matching between the observed
uniform and the correct team under real-world conditions."""

PROMPT_QWEN_7B_GEN = """You are generating a final classification prompt for Qwen2-VL 7B-Instruct.

Qwen2-VL 7B-Instruct is a multimodal instruction-following model that can work with structured text and image inputs. The prompt you create should take advantage of that strength, but it must still remain compact and high-signal.

You will be given:
- Team A uniform feature list (machine-generated)
- Team B uniform feature list (machine-generated)
- Key visual differences between the two uniforms
- optionally, reference images for both teams

These feature lists were written for machine use and contain visually detectable properties of the uniforms.

TASK

Create a prompt that allows Qwen2-VL 7B-Instruct to determine whether an observed uniform matches Team A or Team B, or is a Referee.

REAL-WORLD CONDITIONS

The input image may include:
- motion blur
- glare or reflections
- partial crops (torso only, back only, shorts only)
- occlusion by other players
- fast movement

Because of this, the final prompt should emphasize large, high-contrast, stable features and ignore subtle details.

FEATURE SELECTION RULE

From each team description, select 5 to 8 strong features that:

- are visually dominant
- are easy for a vision-language model to detect
- remain useful under blur, glare, and partial occlusion
- distinguish Team A from Team B

Prioritize, in order:
1) dominant color scheme
2) central text presence and styling
3) number styling (color, outline, placement, but NOT the specific number)
4) thick trim, bold stripes, large color blocks
5) major shorts features (waistband color, bottom bands, side stripes)

Discard:
- player names
- specific jersey numbers
- small logos
- thin details
- low-contrast elements
- features shared by both teams

Use the reference images only to confirm strong features, remove weak or misleading ones, or replace a weak textual feature with a stronger visible one.

PROMPT STRUCTURE RULES

The prompt you create should:
- present Team A and Team B clearly and separately
- organize each team's features in a compact, readable way
- put the strongest distinguishing features first
- include a brief referee description
- guide the model to match the observed uniform against the two feature sets
- rely on visible features, not abstract reasoning

The prompt may use light structure (bullet points), but must remain concise.

CRITICAL CONSTRAINTS

- Do NOT use player-specific identity
- Do NOT use the specific player name
- Do NOT use the specific jersey number
- Only use text/number style, color, outline, and placement if visually distinctive
- Do NOT include long explanations
- Do NOT include chain-of-thought
- Do NOT include unnecessary words
- Do NOT overload the prompt with minor details

TARGET OUTPUT STYLE

The final prompt should compare Team A and Team B using compact feature sets, make the strongest differences easy to see, include referee as a third option, and end with a direct instruction to choose one label.

EXAMPLE OF TARGET OUTPUT

Decide whether the observed uniform matches Team A, Team B, or is a Referee.

Team A:
- black base
- bright green trim at neckline and arm openings
- green-filled numbers with white outline
- green waistband and thick green bands at bottom of shorts
- vertical green side stripes on shorts

Team B:
- white base
- dark side panels
- dark numbers with no outline
- no green accents
- lighter shorts structure

Referee: grey or black-and-white striped shirt, clearly different from both teams.

Output only Team A, Team B, or Referee."""

print("Prompt constants loaded.")

In [ ]:
# ── Unzip jersey images ───────────────────────────────────────────────────────
# Upload jerseys.zip to /content/ via the Colab file browser, then run this cell.
# Expected zip structure: jerseys/<clip>_team0/<img>, jerseys/<clip>_team1/<img>

import zipfile, os

zip_path  = "/content/jerseys.zip"
extract_to = "/content"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        "jerseys.zip not found at /content/jerseys.zip — "
        "upload it via the Colab file browser before running this cell."
    )

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_to)

print(f"Unzipped to {extract_to}/jerseys/")
print("Folders:", sorted(os.listdir(f"{extract_to}/jerseys")))

In [ ]:
import glob

JERSEY_DIR = "/content/jerseys"

# Each team's folder contains one or more images (front, back, shorts, side view, etc.)
# All images in the folder are sent to GPT-4o together for a complete uniform view.
#
# Expected folder structure:
#   /content/jerseys/
#     clip1_easy_team0/   (e.g. front.jpg, back.jpg, shorts.jpg)
#     clip1_easy_team1/
#     clip2_hard_team0/
#     clip2_hard_team1/
#     clip3_edge_team0/
#     clip3_edge_team1/

def get_jersey_images(folder):
    """Return sorted list of all image paths in a jersey folder."""
    exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return sorted(paths)

# team0/team1 folder names in the zip are reversed relative to ground truth team_ids
# swap here so desc0 always matches team_id 0
jersey_dirs = {
    "clip1_easy": (f"{JERSEY_DIR}/clip1_easy_team1", f"{JERSEY_DIR}/clip1_easy_team0"),
    "clip2_hard": (f"{JERSEY_DIR}/clip2_hard_team1", f"{JERSEY_DIR}/clip2_hard_team0"),
    "clip3_edge": (f"{JERSEY_DIR}/clip3_edge_team1", f"{JERSEY_DIR}/clip3_edge_team0"),
}

def get_labels(clip_info):
    """Return (label0, label1) based on special_jersey flags.
    Either team special → neutral labels for both (output format must be consistent).
    Team name can still appear as context in description body for non-special teams.
    """
    special = clip_info["special_jersey"]
    names   = clip_info["team_names"]
    if special["team0"] or special["team1"]:
        return "Team A", "Team B"
    return names[0], names[1]

# Verify folders exist and contain images
missing = []
for clip_name, (dir0, dir1) in jersey_dirs.items():
    for d in (dir0, dir1):
        imgs = get_jersey_images(d) if os.path.isdir(d) else []
        if not imgs:
            missing.append(d)
if missing:
    print("Missing or empty jersey folders — create these in /content/jerseys/:")
    for d in missing:
        print(f"  {d}/")
else:
    total = sum(len(get_jersey_images(d)) for dirs in jersey_dirs.values() for d in dirs)
    print(f"All jersey folders found. {total} images total ready for GPT-4o.")

In [ ]:
# ── Main description generation pipeline ─────────────────────────────────────
# API calls per clip: 2 individual + 1 differences + 1 Florence + 1 Qwen-2B + 1 Qwen-7B = 6
# 3 clips × 6 = 18 calls total. Cost: ~$0.05 per matchup at GPT-4o pricing.
# Multiple images per team are all sent together, giving GPT-4o a complete view.

TEAM_DESCRIPTIONS          = {}   # (desc0, desc1, differences) per clip
TEAM_DESCRIPTIONS_FLORENCE = {}   # complete Florence-2 prompt string per clip
TEAM_DESCRIPTIONS_QWEN_2B  = {}   # complete Qwen2-VL 2B prompt string per clip
TEAM_DESCRIPTIONS_QWEN_7B  = {}   # complete Qwen2-VL 7B prompt string per clip

for clip_name, clip_info in clips.items():
    dir0, dir1     = jersey_dirs[clip_name]
    imgs0          = get_jersey_images(dir0)
    imgs1          = get_jersey_images(dir1)
    label0, label1 = get_labels(clip_info)
    names          = clip_info["team_names"]
    special        = clip_info["special_jersey"]

    # For non-special teams, include team name as context inside description body
    # even when the output label is neutral — gives the model prior knowledge where reliable
    context_0 = f"{names[0]} uniform" if not special["team0"] else "Team A uniform"
    context_1 = f"{names[1]} uniform" if not special["team1"] else "Team B uniform"

    print(f"\n{'='*60}")
    print(f"{clip_name}  |  {label0} ({len(imgs0)} imgs) vs {label1} ({len(imgs1)} imgs)")

    # Steps 1-2: Individual descriptions (all images per team sent together)
    desc0 = call_gpt4o(PROMPT_INDIVIDUAL, imgs0)
    desc1 = call_gpt4o(PROMPT_INDIVIDUAL, imgs1)
    print("  [1/6] [2/6] Individual descriptions done.")

    # Step 3: Differences (all images from both teams together)
    diff = call_gpt4o(PROMPT_DIFFERENCES, imgs0 + imgs1)
    print("  [3/6] Differences description done.")

    # Shared input block for the three generators
    # Florence always uses numeric 0/1 labels (parsed by code); Qwen uses team labels
    florence_input = (
        f"{context_0} description:\n{desc0}\n\n"
        f"{context_1} description:\n{desc1}\n\n"
        f"Key visual differences:\n{diff}\n\n"
        f"Labels to use: 0, 1\n"
        f"ASSIGNMENT RULE — do not override: the first description above ({context_0}) "
        f"must be labeled 0. The second description ({context_1}) must be labeled 1. "
        f"Never swap them regardless of jersey color or brightness.\n"
        f"End the prompt with: Output only 0 or 1."
    )
    gen_input = (
        f"{context_0} description:\n{desc0}\n\n"
        f"{context_1} description:\n{desc1}\n\n"
        f"Key visual differences:\n{diff}\n\n"
        f"Labels to use: {label0}, {label1}, Referee\n"
        f"ASSIGNMENT RULE — do not override: the first description above ({context_0}) "
        f"must be assigned label {label0}. The second description ({context_1}) must be "
        f"assigned label {label1}. Never swap them regardless of jersey color or brightness.\n"
        f"End the prompt with: Output only {label0}, {label1}, or Referee."
    )

    # Step 4: Florence-2 prompt (images confirm/replace weak features)
    florence_prompt = call_gpt4o(
        PROMPT_FLORENCE_GEN + "\n\n" + florence_input,
        imgs0 + imgs1,
        max_tokens=400,
    )
    print("  [4/6] Florence-2 prompt done.")

    # Step 5: Qwen 2B prompt
    qwen_2b_prompt = call_gpt4o(
        PROMPT_QWEN_2B_GEN + "\n\n" + gen_input,
        imgs0 + imgs1,
        max_tokens=600,
    )
    print("  [5/6] Qwen2-VL 2B prompt done.")

    # Step 6: Qwen 7B prompt (more features allowed, light structure ok)
    qwen_7b_prompt = call_gpt4o(
        PROMPT_QWEN_7B_GEN + "\n\n" + gen_input,
        imgs0 + imgs1,
        max_tokens=700,
    )
    print("  [6/6] Qwen2-VL 7B prompt done.")

    TEAM_DESCRIPTIONS[clip_name]          = (desc0, desc1, diff)
    TEAM_DESCRIPTIONS_FLORENCE[clip_name] = florence_prompt
    TEAM_DESCRIPTIONS_QWEN_2B[clip_name]  = qwen_2b_prompt
    TEAM_DESCRIPTIONS_QWEN_7B[clip_name]  = qwen_7b_prompt

print("\nAll descriptions generated successfully.")

In [ ]:
# ── Review generated outputs ─────────────────────────────────────────────────
# Sanity-check all generated descriptions before running models.
# Verify: correct team features, right label names, referee option present.

for clip_name in clips:
    label0, label1 = get_labels(clips[clip_name])
    desc0, desc1, diff = TEAM_DESCRIPTIONS[clip_name]
    print(f"\n{'='*60}")
    print(f"CLIP: {clip_name}  ({label0} / {label1})")
    print(f"\n--- {label0} individual description ---")
    print(desc0)
    print(f"\n--- {label1} individual description ---")
    print(desc1)
    print(f"\n--- Differences ---")
    print(diff)
    print(f"\n--- Florence-2 prompt ---")
    print(TEAM_DESCRIPTIONS_FLORENCE[clip_name])
    print(f"\n--- Qwen2-VL 2B prompt ---")
    print(TEAM_DESCRIPTIONS_QWEN_2B[clip_name])
    print(f"\n--- Qwen2-VL 7B prompt ---")
    print(TEAM_DESCRIPTIONS_QWEN_7B[clip_name])

### Description generation complete

The four dicts are now fully populated:

- **`TEAM_DESCRIPTIONS`** — `(desc0, desc1, differences)` raw building blocks
- **`TEAM_DESCRIPTIONS_FLORENCE`** — complete Florence-2 prompt (2-4 features, highest-signal first)
- **`TEAM_DESCRIPTIONS_QWEN_2B`** — complete Qwen 2B prompt (4-6 features + differences section)
- **`TEAM_DESCRIPTIONS_QWEN_7B`** — complete Qwen 7B prompt (5-8 features, light bullet structure)

All three model prompts include referee as a third output option.
Label strategy was applied automatically from `special_jersey` flags:
- clip1_easy, clip3_edge → neutral labels (special jerseys in use)
- clip2_hard → team names (both standard)

Pipeline: 18 API calls, ~$0.05 per matchup at GPT-4o pricing.

## Section 3 — Model Comparison

The K-Means baseline fails in two distinct ways: (1) it relies on mean RGB color, which collapses when jerseys are similar (clip2_hard, 54%) or lighting varies, and (2) it is sensitive to random initialization on edge cases (clip3_edge). The goal here is to test whether richer visual representations fix these failure modes.

| Model | ID | Approach | Why evaluate |
|-------|----|----------|--------------|
| **SigLIP** | `google/siglip-base-patch16-224` | Embed crops → cluster | Fast, strong image-text alignment, ~350 MB |
| **CLIP** | `openai/clip-vit-base-patch32` | Embed crops → cluster | Widely benchmarked zero-shot baseline |
| **Florence-2** | `microsoft/Florence-2-base` | Prompted classification | Rich visual grounding, region-level reasoning |
| **Qwen2-VL** | `Qwen/Qwen2-VL-2B-Instruct` | Prompted classification | Production stack model; strongest reasoning |

**Embedding strategy (SigLIP, CLIP):** extract per-player crops → get image embeddings → build team prototype centroids from the first GT frame → classify by cosine distance. This replaces mean RGB with a richer feature vector that encodes texture, logo shape, and number patterns — not just color — making it more robust to the similar-color failure mode.

**Prompting strategy (Florence-2, Qwen2-VL):** send each player crop with a structured prompt describing both teams' jersey colors established from the tipoff frame. These models can reason about visual context beyond raw pixel statistics, but have higher latency and memory cost.

### Shared Evaluation Infrastructure

Two functions are defined once and reused across all models:

- **`evaluate_embedding_model`** — builds L2-normalized team prototype centroids from the
  first GT frame, then classifies every valid detection by cosine distance. Tries both
  label orientations (which cluster is team 0?) and keeps the higher accuracy. Also
  measures ms/crop.
- **`evaluate_prompted_model`** — calls a `classify_fn(crop, desc0, desc1) → int` for
  every valid detection, tries both orientations, times the calls.

`TEAM_DESCRIPTIONS` provides human-readable jersey descriptions for the prompted models.

### Building Jersey Descriptions for Prompted Models

Florence-2 and Qwen2-VL receive a natural language description of each team's jersey as
their only calibration signal. The quality of that description directly determines the
ceiling of what the prompted models can achieve — a vague description ("light blue jerseys")
gives the model no more signal than K-Means already has. A precise, angle-invariant feature
list gives it something to look for across front, back, and side views.

**Description generation pipeline:**

1. Pull the official jersey image from the team's merchandise page for the correct season.
   Official merch photos are canonical — correct colors, correct text, no arena lighting
   or motion blur.
2. Feed the image into a frontier VLM (GPT-4o) with the structured prompt below.
   The prompt was designed to produce a *compressed recognition guide*, not a paragraph
   description — dense feature lists that another VLM can match against a crop.
3. Paste the generated output into `TEAM_DESCRIPTIONS` below.

**Description prompt** (crafted iteratively with ChatGPT to optimize for classification):

```
You are analyzing sports uniforms for visual recognition, not for description or marketing.
Your goal is to produce a dense, angle-invariant list of distinctive visual features that
another vision-language model can reliably use to identify the uniform across front, back,
and side views.

Follow these rules strictly:

Focus only on visually distinctive, high-signal features (color blocks, trim, stripes,
logos, text placement, geometry).

Describe relative positions clearly (top, center, sides, bottom, edges, waistband,
neckline, etc.).

Prioritize features that remain visible across multiple angles, especially side and back views.

Treat player names and numbers as variable placeholders. Only describe their style, color,
and placement, not specific values.

Use short, declarative sentences. Each sentence should encode one clear visual rule.

Avoid storytelling, branding language, or unnecessary detail (no fabric discussion, no
adjectives like "sleek" or "modern").

Avoid redundancy. If a feature is already stated, do not restate it.

Emphasize contrast patterns and geometry (for example: black base with green trim at edges,
green bands at bottom of shorts, vertical side stripes).

Include both front and back mappings (for example: front text vs back name/number).

End with a brief summary of the core invariant signals that uniquely identify the uniform.

Output should read like a compressed recognition guide, not a paragraph description.
```

This approach scales to production: a jersey description database keyed by
`(team_id, season, jersey_variant)` means any new matchup only requires generating
descriptions once per jersey, not retraining any model.

---

**How long should the descriptions be? (An open question)**

The intuition that "longer = better" holds for some models but not others.

- **SigLIP / CLIP**: descriptions are not used at all. Irrelevant.

- **Qwen2-VL**: probably better when the description is dense and structured rather than
  padded with fluff — but "definitely better" is too strong. Two real risks apply even to
  clean descriptions: (1) small LLMs tend to over-weight the beginning and end of a prompt
  and under-attend to the middle ("lost in the middle"), so a 15-feature list may not all
  be read; (2) the task output is just "0" or "1" — the model makes one visual judgment, not
  a feature-by-feature checklist, so marginal return on additional features likely flattens
  quickly. The honest position: more precise and specific than K-Means' zero description is
  clearly better; whether 5 features beats 15 features is unknown without testing.

- **Florence-2**: genuinely unclear. Florence-2 was built for structured short-form task
  prompts like `<CAPTION>`, `<OD>`, `<DETAILED_CAPTION>` — not conversational description.
  Its 0.23B text decoder was not trained on long natural-language instructions. A 300-token
  description spanning both teams may cause the model to attend poorly to the specifics and
  effectively ignore most of it, falling back on dominant color alone. It is plausible that
  a short, high-signal description (e.g. "black jersey, green trim, white CELTICS text")
  outperforms the full structured output from the prompt above for Florence-2 specifically.

  This is worth testing empirically: run clip2_hard with both a long and a short description
  and compare Florence-2's accuracy. Qwen2-VL is unlikely to be sensitive to this in the
  same way.

The practical recommendation: use the full description prompt output as-is and treat it as
the baseline. If Florence-2 underperforms expectations, try a stripped-down one-sentence
description for that model only and check whether accuracy improves.


---

**Contrastive prompting**

The classification prompt we use is already contrastive by design: both teams' descriptions
appear in the same prompt, and the model is asked to choose between them. This matters more
than it might seem.

A non-contrastive prompt would describe one team and ask "is this player on this team?" —
but that forces the model to make an absolute judgment against a single reference, which is
harder than a relative one. The contrastive framing — "team 0 looks like X, team 1 looks like
Y, which is this?" — lets the model reason about the *difference* between the two descriptions
rather than matching against either one in isolation. For a task like clip2_hard where light
blue and dark navy are genuinely close, the relative comparison ("which of these two is darker?")
is a much easier question for the model than "does this look light blue?".

This also means description quality interacts with the pairing: a description that would be
ambiguous on its own ("blue jersey") becomes useful when the other team's description is
sufficiently different ("dark navy jersey"). The more a description emphasizes features that
differ *between* the two teams rather than features that are generically true of basketball
jerseys, the more useful it is in the contrastive frame. The description prompt already
encourages this by asking for distinctive and high-contrast features — that instruction is
doing double duty as contrastive guidance.

---

**Qwen2-VL: 2B vs 7B**

Qwen2-VL comes in two sizes relevant here: 2B and 7B parameters. The version in the current
production stack and in this notebook is 2B.

- **2B**: ~4 GB VRAM at fp16, inference at roughly 50-150ms per crop on an H100. Capable
  instruction follower, but limited on fine-grained visual reasoning. On small player crops
  with subtle color differences (exactly the clip2_hard problem), 2B may not have enough
  visual resolution to reliably distinguish light blue from dark navy even with a precise
  description.

- **7B**: ~14 GB VRAM at fp16, roughly 3-5x slower per crop. Meaningfully stronger visual
  reasoning — the larger language model processes the description more fully and the visual
  encoder produces richer tokens. On the hard and edge cases, 7B would likely outperform 2B
  noticeably, particularly when the contrastive description requires subtle color
  discrimination.

Both fit comfortably on an H100 (80 GB). The 7B is viable for a cascade's final escalation
stage where only the hardest cases are routed — if 90% of players are classified confidently
by SigLIP and only the remaining 10% reach Qwen, the 7B latency cost is amortized across
far fewer crops per frame. Whether the accuracy gain justifies the memory and latency cost
is an empirical question this notebook doesn't answer directly, but it is worth noting as a
production lever: swapping 2B for 7B at the top of the cascade is a meaningful upgrade path
if accuracy on hard matchups is the bottleneck.

---

**Three description sets**

Given the above, three prompt variants are used below — one general-purpose, one optimized
for Florence-2's architecture, and one optimized for Qwen2-VL's instruction-following
capacity. Paste the GPT-4o output from each respective prompt into the corresponding dict.


---

**Three-prompt generation strategy**

Rather than generating one description per jersey, three prompts are used:

1. **Individual description — Team 0**: feed the team 0 jersey image alone, ask for
   distinctive visual features. Output is reusable — stored once per jersey and pulled
   for any matchup that team appears in.

2. **Individual description — Team 1**: same, for team 1.

3. **Differences prompt — both jerseys together**: feed both jersey images simultaneously
   and ask specifically what is visually different between them. This is matchup-specific —
   it cannot be reused for a different pairing — but it does the contrastive reasoning
   offline so the VLM at inference time does not have to reconstruct it from two separate
   descriptions.

The differences prompt is the most valuable for hard matchups. For clip2_hard, asking GPT-4o
"what is visually different between these two jerseys?" forces it to resolve the light blue
vs dark navy distinction explicitly — producing output like "team 0 is noticeably lighter and
more saturated, team 1 is darker and more muted" — which is exactly the comparison Qwen2-VL
needs to make at inference time. Generating that contrast once is better than hoping the model
reconstructs it from two independent descriptions mid-classification.

The ideal combined description passed to the prompted models is:
`[team 0 individual] + [team 1 individual] + [key differences]`

This maps directly to the three prompts.

**On compute cost:** this is text generation done once per game at tipoff, not once per frame
or per player. Even using GPT-4o for all three prompts, the cost is three API calls per game —
negligible at any scale. At 1000 games per day this is 3000 API calls total, which at GPT-4o
pricing is a rounding error compared to the inference cost of running Qwen2-VL on every
uncertain player crop. There is no reason to optimize or cut corners here.

**Which model for the differences prompt?**

The differences task requires multi-image input and fine-grained color discrimination. Honest
ranking:

- **GPT-4o** — the natural choice given you are already using it for prompts 1 and 2.
  Handles multi-image prompts well and produces structured output reliably. Use this.

- **Gemini 1.5 Pro / 2.0** — worth knowing about. Gemini has shown strong cross-image visual
  comparison ability, and the "what's different between these two images" framing is close to
  tasks it was specifically evaluated on. If GPT-4o's differences description feels vague on
  clip2_hard specifically, Gemini is the first alternative to try.

- **Claude 3.5 Sonnet** — comparable to GPT-4o on structured output generation, supports
  multi-image input. No strong reason to switch from GPT-4o for this task.

Avoid smaller or open-source vision models for the differences prompt — fine-grained
cross-image color comparison is precisely where they tend to produce unreliable output, and
the whole point of the differences prompt is to resolve subtle distinctions that cheaper
approaches miss.


---

**Prompt refinements — v2**

Two changes were made to the original prompt:

*Ordering added in two places.* "Order features from most visually dominant to least" appears
in both REQUIREMENTS and OUTPUT FORMAT. The original prompt produced a flat list with no
guaranteed order — GPT-4o would sometimes front-load minor details (small logo placement)
before dominant features (base color). This matters because Florence-2 likely reads the first
few lines most attentively and down-weights the rest. Ordering by visual dominance ensures
the most diagnostic features are read first regardless of which model consumes the output.

*"Do NOT compare to other uniforms" removed.* This constraint was added conservatively but
is unnecessary — the prompt already specifies a single image input, frames the task as
feature extraction not classification, and prohibits reasoning and explanations. A model
following these instructions has no basis to introduce comparisons. Keeping redundant
constraints risks confusing the model about what is actually required.

```
You are analyzing a single basketball uniform image.

Your output will be consumed by other AI models, including Florence-2, Qwen2-VL, and similar vision-language models. These models have limited attention and rely on short, high-signal, visually detectable features. You must write strictly for machine consumption.

---

TASK

Produce a list of distinctive, machine-detectable visual features that define the uniform.

---

REQUIREMENTS

- Use short, declarative sentences.
- Each sentence must describe exactly ONE visual feature.
- Order features from most visually dominant to least.
- Prefer features that are:
  - visually obvious
  - high-contrast
  - geometrically clear
  - consistently detectable by a model

Focus on:
- Dominant color(s) and color combinations
- Text presence and styling (color, outline, placement, size)
- Number styling (color, outline, placement), but NOT the specific number
- Trim and edge boundaries (neckline, arm openings)
- Stripes, bands, and geometric patterns
- Shorts features (waistband color, side stripes, bottom bands)

Strongly prefer:
- Features visible from multiple angles (front, back, side)
- Features that remain stable under partial occlusion

---

CRITICAL CONSTRAINTS

- Do NOT use the specific player name or number as a feature
- Do NOT rely on any identity that changes from player to player
- Only describe the STYLE and PLACEMENT of text and numbers

- Do NOT classify the team
- Do NOT include reasoning or explanations
- Do NOT include fabric, material, or stylistic commentary
- Avoid redundancy
- Avoid vague or subjective language

---

OUTPUT FORMAT

- A flat list of feature statements
- One sentence per line
- No grouping, no sections
- Order from most visually dominant feature to least

---

EXAMPLE OUTPUT

Black base uniform.
Green trim around neckline and arm openings.
White team name centered on chest.
Numbers have green fill with white outline.
Green waistband on shorts.
Thick green bands at the bottom of shorts.
Vertical green side stripes on shorts from waistband to hem.

---

Your goal is to produce a dense, consistent feature list that downstream AI systems can reliably use to recognize this uniform across different views.
```


---

**Florence-2 prompt (generated by ChatGPT from the individual descriptions)**

This prompt is generated once per matchup by feeding both team descriptions (and optionally
both jersey reference images) into ChatGPT. The output is the actual string passed to
Florence-2 at classification time — not another description layer, but the final prompt.

Key design decisions this prompt enforces:

- **2-4 features per team hard limit.** Florence-2's small decoder cannot reliably attend
  to longer feature lists. Forcing GPT-4o to select only the strongest features means the
  ones that matter most appear in the prompt rather than being buried.

- **"Do not mention features shared by both teams."** This is the most important instruction.
  Shared features add no classification signal and waste Florence-2's attention budget. By
  removing them at generation time, the prompt only contains contrastive information.

- **Environment constraints drive feature selection.** The prompt instructs GPT-4o to discard
  features likely to disappear under motion blur, glare, or partial occlusion. This aligns
  the feature set with what is actually detectable in game footage rather than what looks
  distinctive in a studio photo.

- **Labels must be "0" and "1", not "Team A" and "Team B".** The generated prompt uses
  numeric labels to match the response parser in the code (`"0" in response`). If GPT-4o
  uses "Team A"/"Team B" instead, the parser silently returns the wrong answer for every
  crop. Adding this as an explicit constraint keeps the pipeline consistent.

- **"Output only 0 or 1"** forces a single-token response. Florence-2 tends toward verbose
  outputs; constraining the response format reduces the chance of the parser failing on a
  long sentence that happens to contain both digits.

The prompt to paste into ChatGPT (with both descriptions already generated):

```
You are generating a final prompt for Florence-2.

Florence-2 is a prompt-based vision-language model that works best with short, concrete,
caption-like instructions and does not benefit much from long reasoning-heavy prompts.
The final prompt must therefore be extremely short and feature-first.

Inputs you will receive:
- Team 0 machine-readable uniform description
- Team 1 machine-readable uniform description
- optionally, reference images for Team 0 and Team 1

Task:
Create the shortest possible Florence-2 prompt that lets the model choose between Team 0
and Team 1 in a real basketball image.

Important environment constraints:
- real game footage may contain motion blur
- glare or bright reflections
- partial crops
- occlusion by other players
- fast movement

Because of this, prefer only large, high-contrast, easy-to-detect features. Ignore minor
or subtle details.

Feature selection rules:
- choose only 2 to 4 features per team
- prefer dominant colors, large text styling, major number styling, thick trim, bold
  stripes, large shorts bands, or major color blocking
- do not use player names
- do not use the specific jersey number
- only use text or number style, color, outline, and placement if visually distinctive
- discard any feature that is small, thin, low-contrast, or likely to disappear under
  blur or glare
- do not mention features shared by both teams — shared features add no signal and waste
  the model's attention
- if images are provided, use them to confirm or replace the weakest features in the
  descriptions; if not, use the descriptions alone

Prompt style rules:
- keep the final Florence-2 prompt extremely short
- put the strongest distinguishing feature first
- use simple concrete visual words
- do not include explanations
- do not include sections
- do not include reasoning steps
- use the labels 0 and 1 (not "Team A"/"Team B") — the response is parsed by code that
  looks for these exact characters
- end with: Output only 0 or 1.

Target form:
Use feature-to-label mapping, not abstract label discussion.

Example target style:
Black uniform with bright green trim and green shorts bands is 0.
White uniform with dark side panels and dark numbers is 1.
Which label matches this player crop? Output only 0 or 1.

Goal:
Produce the shortest Florence-2 prompt that preserves the strongest real-world visual
separation between the two teams.
```


---

**Label strategy: team names vs neutral labels**

The generated prompts can use either actual team names ("Celtics", "Heat") or neutral labels
("Team A", "Team B") as the classification targets. The choice matters because instruction-tuned
VLMs like Qwen2-VL have prior knowledge about NBA teams from training data — and that knowledge
can either help or hurt depending on what the players are actually wearing.

**Use actual team names when standard jerseys are worn.**
If the Celtics are wearing their standard away kit, the model's prior knowledge that "Celtics
wear black and green" is accurate and adds signal on top of the description. The name acts as
a soft prior that reinforces the visual features. Description plus name is stronger than either
alone.

**Use neutral labels ("Team A", "Team B") when special jerseys are worn.**
Christmas jerseys, throwbacks, city editions, and alternate kits break the model's memorized
associations. If Qwen "knows" the Cavs wear wine and gold but they are wearing a navy throwback
for a Christmas game, using the name "Cavaliers" in the prompt actively misleads the model —
it will fight against the description rather than use it. Neutral labels force the model to
rely entirely on the visual description you provide.

This maps to the clips in this notebook:

| Clip | Team 0 | Team 0 jersey | Team 1 | Team 1 jersey | Output labels |
|------|--------|---------------|--------|---------------|---------------|
| clip1_easy | Celtics | Statement (black) — uncertain | Heat | Standard | Neutral — either team special triggers neutral for both |
| clip2_hard | Spurs | Standard | Grizzlies | Standard | Team names |
| clip3_edge | Cavaliers | Christmas throwback | Knicks | Christmas | Neutral |

**Rule:** track `special_jersey` per team. If **either** team has a special jersey, use neutral
output labels ("Team A", "Team B") for both — the output format must be consistent. For the
team with a standard jersey, the prompt generator can still include the team name as context
inside the description body to leverage prior knowledge without it affecting the output label.

**Production implication:** the system needs to know at prompt generation time whether special
jerseys are in use. This can be determined from game metadata (schedule, jersey variant flag)
before tipoff — another reason the description generation step happens once per game rather
than per frame.

**Parsing:** the response parser needs to handle both formats. A clip using team names will
produce responses like "Celtics" or "Heat"; a clip using neutral labels will produce "Team A"
or "Team B". Both are unambiguous and easy to parse — and both are cleaner than numeric
labels where "0" or "1" can appear anywhere in a verbose response.


---

**Qwen2-VL prompt generator**

Same pipeline as Florence-2: feed both team descriptions (and optionally both jersey images)
into ChatGPT with the prompt below. The output is the actual string passed to Qwen2-VL at
classification time, stored in `TEAM_DESCRIPTIONS_QWEN`.

Key differences from the Florence-2 prompt generator:

- **4-6 features per team instead of 2-4.** Qwen2-VL is an instruction-tuned model with
  genuine language understanding — it can use a richer feature set than Florence-2's small
  decoder. More features give it more anchors, especially on subtle cases like clip2_hard.

- **Differences description as a third input.** The Qwen prompt generator accepts the output
  of the differences prompt (prompt 3) as an optional third input and places it after both
  team descriptions as a "key distinguishing signals" section. This is the most valuable
  contrastive signal and Qwen, unlike Florence-2, has the capacity to use it.

- **Label format depends on `special_jersey` flags.** If either team has `special_jersey:
  True`, use neutral labels ("Team A", "Team B") for both. If both are standard, use actual
  team names. Either way, include "Referee" as a third output option. The output format must
  be consistent — you cannot mix a team name with a neutral label in the same prompt. For the
  non-special team in a mixed game, include the team name as context inside the description
  body even when the output label is neutral.

- **"Output only [label0], [label1], or Referee."** Single-token or two-word output.
  Qwen2-VL is instruction-tuned well enough to follow this reliably, but raw response logging
  in the code will surface any cases where it does not.

- **No chain-of-thought.** At 2B parameters, CoT tends to produce rambling output rather than
  useful intermediate steps. Suppress it explicitly.

```
You are generating a final classification prompt for Qwen2-VL 2B.

Qwen2-VL 2B is a small vision-language model. It can follow structured instructions but
loses attention in the middle of long prompts. It performs best when given compact,
high-signal, well-ordered information.

You will be given:
- Team 0 uniform feature list (machine-generated)
- Team 1 uniform feature list (machine-generated)
- optionally, a differences description comparing the two uniforms
- optionally, reference images for both teams

These feature lists already contain visually detectable properties.

---

TASK

Create a prompt that allows Qwen2-VL 2B to determine whether an observed uniform matches
Team 0, Team 1, or a Referee.

---

REAL-WORLD CONDITIONS

The input image may include:
- motion blur
- glare or reflections
- partial crops (torso only, back only, shorts only)
- occlusion by other players

Because of this:
- small or subtle features are unreliable
- large, high-contrast features are most important

---

FEATURE SELECTION RULE

From each team, select 4 to 6 features that:

- are visually dominant
- are high-contrast and easy to detect
- remain visible under blur, glare, and occlusion
- help distinguish the two teams from each other

Prioritize:
1) dominant color scheme
2) central text presence and styling
3) number styling (color and outline, NOT the specific number)
4) strong trim, bold stripes, or large color blocks
5) major shorts features (waistband color, large bands, side stripes)

Discard:
- small logos
- thin details
- low-contrast elements
- features shared by both teams

If a differences description is provided, include its key signals as a short final section
after both team descriptions. Order the differences by visual importance.

If reference images are provided, use them to confirm or replace weak features from the
descriptions.

---

LABEL RULES

Use the labels provided to you. Do not change them. The labels will be either:
- actual team names (e.g. "Spurs", "Grizzlies") when both teams wear standard jerseys
- neutral labels ("Team A", "Team B") when either team wears a special or alternate jersey

Always include "Referee" as a third output option, described as: grey or black and white
striped shirt, clearly different from both team uniforms.

For a non-special team in a mixed game (one standard, one special), include the team name
as context inside the description body even when the output label is neutral.

---

PROMPT STRUCTURE RULES

The prompt you create must:

- Present both team descriptions clearly and separately
- Use short, declarative feature phrases
- Order features from most diagnostic to least
- Place the most important features early
- Include a brief referee description
- End with a direct instruction to choose one label

---

CRITICAL CONSTRAINTS

- Do NOT use player names or specific numbers
- Only use style and placement of text and numbers
- Do NOT include long explanations
- Do NOT include chain-of-thought
- Do NOT include unnecessary words
- Keep total prompt length moderate and dense

---

OUTPUT FORMAT

The final prompt must end with exactly:
Output only [label0], [label1], or Referee.

where [label0] and [label1] are the labels provided to you.

---

EXAMPLE TARGET OUTPUT (neutral labels)

Decide which team this player belongs to, or identify them as a referee.

Team A: black base, bright green trim at neckline and arms, green waistband and thick green
bands at bottom of shorts, vertical green side stripes, white team name centered on chest,
green-filled numbers with white outline.

Team B: black base, red and white trim, white team name centered on chest, red and white
numbers, no green anywhere.

Key difference: Team A has green accents throughout. Team B has red and white accents.

Referee: grey or black and white striped shirt, clearly different from both teams.

Output only Team A, Team B, or Referee.

---

GOAL

Produce a compact, well-ordered prompt that maximizes correct matching between the observed
uniform and the correct team under real-world conditions.
```

*Jersey description dicts (`TEAM_DESCRIPTIONS`, `TEAM_DESCRIPTIONS_FLORENCE`,
`TEAM_DESCRIPTIONS_QWEN`) are populated automatically by Section 1.5 above.
Run that section before executing any cells in this section.*

In [ ]:
import time
import torch
from PIL import Image
from src.utils import crop_player

def evaluate_embedding_model(embed_fn, frames_dict, gt_dict, clips):
    """
    Embedding-based team classification.

    For each clip:
      1. Find the first GT frame with >=2 valid non-referee bboxes.
         Embed each player crop, average embeddings per team -> prototype centroids.
      2. Classify all valid non-referee labels by cosine distance (1 - dot product).
      3. Try both label orientations, take max accuracy.
      4. Time embed_fn per crop.

    Returns:
        dict: clip_name -> {accuracy, n_detections, ms_per_crop}
    """
    results = {}
    for clip_name in clips:
        frames  = frames_dict[clip_name]
        gt_data = gt_dict[clip_name]

        # Build prototype centroids from first usable GT frame
        prototypes = {0: [], 1: []}
        fit_done = False
        for entry in gt_data:
            player_labels = [l for l in entry["labels"] if l["valid"] and l["team_id"] != -1]
            teams_present = set(l["team_id"] for l in player_labels)
            if len(teams_present) < 2:
                continue
            frame = frames[entry["frame_idx"]]
            for label in player_labels:
                crop = crop_player(frame, label["bbox"])
                emb  = embed_fn(crop)
                prototypes[label["team_id"]].append(emb)
            fit_done = True
            break

        assert fit_done, f"No GT frame with both teams found for {clip_name}"
        proto = {t: np.mean(np.stack(embs), axis=0) for t, embs in prototypes.items()}
        proto = {t: v / np.linalg.norm(v) for t, v in proto.items()}

        # Predict + time
        preds, gt_labels, times = [], [], []
        for entry in gt_data:
            frame = frames[entry["frame_idx"]]
            for label in entry["labels"]:
                if not label["valid"] or label["team_id"] == -1:
                    continue
                crop = crop_player(frame, label["bbox"])
                t0  = time.time()
                emb = embed_fn(crop)
                times.append((time.time() - t0) * 1000)
                d0 = 1 - float(np.dot(emb, proto[0]))
                d1 = 1 - float(np.dot(emb, proto[1]))
                preds.append(0 if d0 < d1 else 1)
                gt_labels.append(label["team_id"])

        preds = np.array(preds)
        gt    = np.array(gt_labels)
        acc   = max(float(np.mean(preds == gt)), float(np.mean((1 - preds) == gt)))
        results[clip_name] = {
            "accuracy":      round(acc, 3),
            "n_detections":  len(preds),
            "ms_per_crop":   round(float(np.mean(times)), 2),
        }
    return results


def evaluate_prompted_model(classify_fn, team_descriptions, frames_dict, gt_dict, clips):
    """
    Prompt-based team classification.

    For each clip:
      1. classify_fn(crop_bgr, team0_desc, team1_desc) -> int (0 or 1)
         team_descriptions values may be (desc0, desc1) or (desc0, desc1, differences).
         If a differences string is present it is appended to both descriptions.
      2. Collect predictions over all valid non-referee labels.
      3. Try both orientation assignments, take max accuracy.
      4. Time classify_fn per call.

    Returns:
        dict: clip_name -> {accuracy, n_detections, ms_per_crop}
    """
    results = {}
    for clip_name in clips:
        frames  = frames_dict[clip_name]
        gt_data = gt_dict[clip_name]
        descs   = team_descriptions[clip_name]
        desc0, desc1 = descs[0], descs[1]
        if len(descs) == 3 and descs[2]:
            desc0 = desc0 + " " + descs[2]
            desc1 = desc1 + " " + descs[2]

        preds, gt_labels, times = [], [], []
        for entry in gt_data:
            frame = frames[entry["frame_idx"]]
            for label in entry["labels"]:
                if not label["valid"] or label["team_id"] == -1:
                    continue
                crop = crop_player(frame, label["bbox"])
                t0   = time.time()
                pred = classify_fn(crop, desc0, desc1)
                times.append((time.time() - t0) * 1000)
                preds.append(pred)
                gt_labels.append(label["team_id"])

        preds = np.array(preds)
        gt    = np.array(gt_labels)
        acc   = max(float(np.mean(preds == gt)), float(np.mean((1 - preds) == gt)))
        results[clip_name] = {
            "accuracy":     round(acc, 3),
            "n_detections": len(preds),
            "ms_per_crop":  round(float(np.mean(times)), 2),
        }
    return results

### 3.1 SigLIP — Visual Embedding + Cosine Clustering

**Model:** `google/siglip-base-patch16-224`  
**Strategy:** extract per-player crop embeddings → build L2-normalized team centroid prototypes
from the first GT frame → classify by cosine distance

SigLIP was trained with a sigmoid image-text contrastive objective on large-scale web data,
which means its embedding space encodes visual semantics beyond raw pixel values — texture,
shape, logo structure, and jersey number patterns all contribute to the representation. This
is the fundamental difference from K-Means on mean RGB: where K-Means collapses each player
to a single `[R, G, B]` triplet and fails when two teams' colors average similarly, SigLIP
operates in a 768-dimensional space where there may be axes that separate the teams even when
mean color alone does not.

**What we expect per clip:**

- **clip1_easy** (Celtics vs Heat): K-Means already scores ~87–89% here, so SigLIP should hold
  or improve slightly. White/green vs black/red is a visually obvious separation. More
  interesting is whether the result is *stable* — unlike K-Means, SigLIP's prototype
  computation is deterministic once the first-frame embeddings are extracted, so we should see
  the same number every run.
- **clip2_hard** (Spurs vs Grizzlies): This is the decisive test. Light blue vs dark navy are
  different colors, but K-Means collapsed at 54% because mean RGB isn't sensitive enough. The
  question is whether 768-dimensional SigLIP embeddings separate them better. If SigLIP
  significantly outperforms K-Means here, it confirms that richer features solve the color
  similarity problem. If it also fails, we have a harder challenge — the jerseys may simply
  look similar at multiple levels of representation.
- **clip3_edge** (Cavs vs Knicks): The bimodal K-Means behavior on this clip was caused by
  non-deterministic initialization, not inherently ambiguous features. Navy vs white/orange
  *should* be distinguishable. SigLIP, being deterministic, will land on one answer and stay
  there. The question is whether it lands on the good answer (~90%) or the bad one (~81%),
  which depends on whether the first GT frame gives representative prototypes for both teams.

In [ ]:
from src.utils import load_siglip_model, extract_siglip_embedding
from src.config import MODEL_SIGLIP

device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
siglip_model, siglip_processor = load_siglip_model(device)
siglip_mem_mb = torch.cuda.max_memory_allocated() / 1e6

def get_siglip_embedding(crop_bgr):
    image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    inputs = siglip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = siglip_model.get_image_features(**inputs)
    if isinstance(outputs, torch.Tensor):
        emb = outputs.squeeze().cpu().numpy()
    else:
        emb = outputs[0].squeeze().cpu().numpy()
    while emb.ndim > 1:
        emb = emb.mean(axis=0)
    return emb / np.linalg.norm(emb)

siglip_results = evaluate_embedding_model(get_siglip_embedding, frames_dict, gt_dict, clips)

print(f"SigLIP peak GPU memory: {siglip_mem_mb:.0f} MB\n")
for name, r in siglip_results.items():
    print(f"  {name}: {r['accuracy']:.1%}  ({r['n_detections']} det, {r['ms_per_crop']:.1f} ms/crop)")

#### SigLIP — Results and Deliberation

*(Fill in after running — template below. Replace bracketed text with actual numbers.)*

---

The clip2_hard result is the most telling number in this table. If SigLIP beats the 54%
K-Means baseline here, it means the color-similarity failure mode is a feature-richness
problem — 768-dimensional visual embeddings capture something that mean RGB misses, likely
texture or the relative darkness of the jerseys independent of hue. If SigLIP also falls
near 50–55%, we have stronger evidence that light blue vs dark navy are genuinely difficult
at the embedding level, not just the pixel level. That would push us harder toward prompted
models that can reason about color descriptions explicitly.

On clip1_easy and clip3_edge, the more interesting observation is *stability* rather than
raw accuracy. K-Means was bimodal on clip3 — 90.6% or 81.2% on alternating runs. SigLIP
should produce the same number every time. If that number is near 90%, the bimodal problem
was purely an initialization artifact. If it locks in at 81% or lower, it suggests the
first-frame prototypes may not capture a fully representative embedding for one of the teams,
and a strategy like averaging over multiple early frames would help.

One thing to watch: SigLIP was not trained on sports imagery specifically. The embeddings are
tuned for web-scale image-text pairs, so a jersey crop that shows mainly a colored rectangle
with a number printed on it is a somewhat unusual input. It is plausible that jersey
classification works well despite this — general visual features are often sufficient — but
if results are underwhelming, domain-specific fine-tuning would be the natural next step.

### 3.2 CLIP — Visual Embedding + Cosine Clustering

**Model:** `openai/clip-vit-base-patch32`  
**Strategy:** same as SigLIP — embed crops, build prototypes from first GT frame, classify
by cosine distance

CLIP is the older predecessor to SigLIP. Both are contrastive image-text models that produce
a joint embedding space, but they differ in training objective and scale: CLIP uses InfoNCE
loss (softmax over negatives in the batch), while SigLIP uses a sigmoid binary loss that
treats each image-text pair independently and scales better to large batch sizes. In practice
SigLIP tends to produce stronger zero-shot representations, but the gap varies by task.

The main reason to run CLIP is not because we expect it to win — it almost certainly won't
outperform SigLIP — but because it is the de facto standard reference point. Any paper or
technical report that introduces a new visual representation model benchmarks against CLIP
ViT-B/32. Including it here means our results table can be read directly against published
benchmarks elsewhere.

**What we expect:** CLIP should perform similarly to SigLIP on clip1 and clip3, where the
separation is clear enough that either model should find it. The most interesting comparison
is on clip2_hard: if SigLIP improved over K-Means there but CLIP did not, it would suggest
the SigLIP training objective specifically helps with fine-grained color separation. If both
improve or both fail similarly, the result comes down to feature dimensionality and pre-training
scale rather than the loss function.

On latency: CLIP ViT-B/32 has a smaller patch size than SigLIP base-patch16, which affects
throughput differently depending on input resolution. At typical jersey crop sizes, CLIP may
be marginally faster or slower — worth measuring directly rather than assuming.

In [ ]:
from transformers import CLIPProcessor, CLIPModel

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_mem_mb    = torch.cuda.max_memory_allocated() / 1e6

def get_clip_embedding(crop_bgr):
    image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
    if isinstance(outputs, torch.Tensor):
        emb = outputs.squeeze().cpu().numpy()
    else:
        emb = outputs[0].squeeze().cpu().numpy()
    while emb.ndim > 1:
        emb = emb.mean(axis=0)
    return emb / np.linalg.norm(emb)

clip_results = evaluate_embedding_model(get_clip_embedding, frames_dict, gt_dict, clips)

print(f"CLIP peak GPU memory: {clip_mem_mb:.0f} MB\n")
for name, r in clip_results.items():
    print(f"  {name}: {r['accuracy']:.1%}  ({r['n_detections']} det, {r['ms_per_crop']:.1f} ms/crop)")

#### CLIP — Results and Deliberation

*(Fill in after running — template below. Replace bracketed text with actual numbers.)*

---

The CLIP vs SigLIP comparison on clip2_hard is the core question here. Three interesting
scenarios:

**Scenario A — SigLIP improves, CLIP does not:** The sigmoid training objective in SigLIP
specifically helps fine-grained visual discrimination. SigLIP's training procedure encourages
the model to correctly assign each image-text pair independently, which may produce tighter
clusters for visually similar categories. This would give us a concrete technical argument
for preferring SigLIP as the embedding backbone.

**Scenario B — Both improve over K-Means:** The failure was a feature-richness problem, and
both models solve it. The more nuanced question is then *how much* each improves — a large
gap still favors SigLIP, while near-identical results suggest any contrastive visual model
is sufficient and CLIP's broader ecosystem support may favor it for integration.

**Scenario C — Neither improves significantly:** The hardest case. Light blue vs dark navy
are genuinely difficult to separate even in rich visual feature spaces. This would suggest
that jersey color similarity is not fully resolved by better embeddings alone, and that
text-guided reasoning (Florence-2, Qwen2-VL) or external signals (roster lookup, jersey
number OCR) are needed for this failure mode.

Also worth comparing ms/crop between CLIP and SigLIP. Embedding models are the cheapest
stage in a cascade architecture, so latency differences here are small in absolute terms —
but at 1000 games × 30 fps × ~10 players/frame, even a 5ms difference per crop compounds
to real cost at scale.

### 3.3 Florence-2 — Direct Prompted Classification

**Model:** `microsoft/Florence-2-base`  
**Strategy:** send each player crop with a natural language prompt describing both teams'
jerseys → parse the model's text output for "0" or "1"

Florence-2 is a different kind of model from SigLIP and CLIP. Rather than producing a
generic embedding for downstream tasks, it is a vision-language model that takes structured
task prompts and generates text outputs. It was built specifically for dense visual
understanding tasks — grounding, captioning, detection, segmentation — using a large synthetic
dataset called FLD-5B. That emphasis on region-level reasoning is why it's interesting for
player classification: we're sending cropped individual players, not full frames, so the
model needs to reason about what a specific crop shows.

**The classification approach:** we describe both teams in the prompt and ask the model to
output "0" or "1". This is a fundamentally different signal than the embedding models: instead
of asking "which pre-computed prototype does this embedding resemble?", we're asking "given
that team 0 wears X and team 1 wears Y, which team is this player on?". The model can
potentially use its language understanding to reason about the color description more precisely
than pixel-level similarity allows.

**Why this could help on clip2_hard:** the K-Means failure on Spurs vs Grizzlies (54%) was
caused by light blue and dark navy colors averaging similarly in RGB space. If Florence-2
receives a crop and the instruction "team 0 wears light blue, team 1 wears dark navy", it
might be able to make a finer visual distinction — the model understands that "dark navy" is
much darker than "light blue" and can look for brightness/saturation differences rather than
hue alone.

**Why it might not:** Florence-2-base is a 0.27B parameter vision encoder + 0.23B parameter
text decoder. The visual encoder may not have sufficient resolution to distinguish subtle
jersey color differences in a heavily compressed player crop. Additionally, the model may
"hallucinate" — responding with "0" regardless of the image if the prompt formatting is not
exactly what it expects. We'll log a few raw responses to catch this.

**Expected latency:** significantly higher than the embedding models. Each call involves a
full autoregressive decode, not just a forward pass through a ViT encoder.

In [ ]:
# ── Florence-2 setup ─────────────────────────────────────────────────────────
# Requires RUN_MODEL = "florence" (transformers==4.44.2).

florence_results = None

if RUN_MODEL == "florence":
    from transformers import AutoProcessor, AutoModelForCausalLM

    FLORENCE_MODEL_ID = "microsoft/Florence-2-base"

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    florence_processor = AutoProcessor.from_pretrained(
        FLORENCE_MODEL_ID, trust_remote_code=True
    )
    florence_model = AutoModelForCausalLM.from_pretrained(
        FLORENCE_MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16
    ).to(device).eval()

    florence_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0
    print(f"Florence-2 loaded on {device} ({florence_mem_mb:.0f} MB)")

    def classify_with_florence(crop_bgr, team0_desc, team1_desc, clip_name=None):
        """Classify a player crop using Florence-2."""
        image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        if clip_name and clip_name in TEAM_DESCRIPTIONS_FLORENCE:
            prompt = TEAM_DESCRIPTIONS_FLORENCE[clip_name]
        else:
            prompt = (
                f"<CLASSIFICATION> 0 wears {team0_desc}. "
                f"1 wears {team1_desc}. "
                "Which label matches this player crop? Output only 0 or 1."
            )
        inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = florence_model.generate(**inputs, max_new_tokens=10)
        response = florence_processor.decode(output_ids[0], skip_special_tokens=True).strip()
        print(f"  raw: {response!r}", end="  ")
        resp_lower = response.lower()
        if "referee" in resp_lower:
            return -1
        if resp_lower.startswith("0") or ("0" in resp_lower and "1" not in resp_lower):
            return 0
        if resp_lower.startswith("1") or ("1" in resp_lower and "0" not in resp_lower):
            return 1
        print("UNPARSEABLE", end="  ")
        return -2

    florence_results = evaluate_prompted_model(
        classify_with_florence, TEAM_DESCRIPTIONS, frames_dict, gt_dict, clips
    )
    print(f"\nFlorence-2 peak GPU memory: {florence_mem_mb:.0f} MB\n")
    for name, r in florence_results.items():
        print(f"  {name}: {r['accuracy']:.1%}  ({r['n_detections']} det, {r['ms_per_crop']:.1f} ms/crop)")
else:
    print("⏭ Skipping Florence-2 (RUN_MODEL != 'florence')")
    print("  To run Florence-2: set RUN_MODEL = 'florence' in Cell 2, restart runtime, rerun all.")

#### Florence-2 — Results and Deliberation

*(Fill in after running — template below. Replace bracketed text with actual numbers.)*

---

The most diagnostic question for Florence-2 is whether it beats the embedding models on
clip2_hard. If it does, the gap between embedding-based and prompting-based approaches is
real: natural language descriptions allow the model to apply its understanding of "dark navy
vs light blue" in a way that unsupervised clustering over pixel features cannot. If it doesn't,
either the model is not effectively using the color description (check the raw responses for
hallucination) or the crop resolution is too low for fine-grained color discrimination at any
level.

One failure mode to watch specifically is "response collapse" — where the model always outputs
the same answer regardless of the image. The raw response sample logged above gives an early
signal. A well-formed output should be short ("0" or "1" with maybe a few extra words). If the
response is a full sentence containing both digits, or always starts with "0", the accuracy
number is meaningless and the prompt needs to be redesigned.

The latency number here is more important than for the embedding models. Florence-2 running at
e.g. 200ms/crop would make it 10–20x slower than SigLIP. For a cascade architecture, that's
acceptable if Florence-2 is only called for uncertain cases — maybe 10–20% of players — but
completely impractical as the primary classifier at 30 fps. The latency measurement directly
informs how aggressively we can route players to this stage.

### 3.4 Qwen2-VL — Direct Prompted Classification

**Model:** `Qwen/Qwen2-VL-2B-Instruct`  
**Strategy:** same prompt-based approach as Florence-2, using Qwen's chat template format

Qwen2-VL is the model already deployed in the production stack, which makes this subsection
different from the others. For SigLIP, CLIP, and Florence-2, the question is "is this model
good enough to consider adopting?". For Qwen2-VL, the question is "how well does the
system we already have actually perform, and where does it need help?".

**The core hypothesis:** Qwen2-VL's instruction-following capability and broader visual
reasoning should make it the most accurate model in this comparison — but the 2B parameter
version may struggle with subtle visual distinctions that require high spatial resolution.
There is a real tension between "can reason about jersey descriptions in text" and "can
actually see fine color differences in a small, compressed player crop". Large models reason
better; small models are cheaper. The 2B version is optimized for production latency, not
peak accuracy.

**What makes Qwen2-VL potentially uniquely valuable:** jersey number OCR. The embedding
models (SigLIP, CLIP) have no mechanism for reading the number off a jersey. A prompted
model that can read "34" from a jersey and match it to a known roster would unlock a
completely different level of accuracy — especially in edge cases where colors are ambiguous.
This notebook doesn't test the OCR pathway directly, but the qualitative responses from
Qwen2-VL will give us a sense of whether the model is attending to jersey text at all.

**What we expect on the hard clip:** on clip2_hard (Spurs vs Grizzlies), Qwen2-VL 2B receives
the same jersey description as Florence-2. Whether it outperforms Florence-2 here reflects
model scale and instruction-following quality rather than the approach. If both fail, the
color-discrimination bottleneck applies even to prompted VLMs at this parameter count.

In [ ]:
# ── Qwen2-VL setup ───────────────────────────────────────────────────────────
# Requires RUN_MODEL = "qwen" (transformers>=4.49).

qwen_results = None

if RUN_MODEL == "qwen":
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor as QwenAutoProcessor

    QWEN_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    qwen_processor = QwenAutoProcessor.from_pretrained(QWEN_MODEL_ID)
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID, torch_dtype="auto"
    ).to(device).eval()

    qwen_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0
    print(f"Qwen2-VL loaded on {device} ({qwen_mem_mb:.0f} MB)")

    def classify_with_qwen(crop_bgr, team0_desc, team1_desc):
        """Classify a player crop using Qwen2-VL with jersey descriptions."""
        image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        prompt = (
            f"Team 0 wears {team0_desc}. "
            f"Team 1 wears {team1_desc}. "
            "Look at this basketball player image. "
            "Which team does this player belong to? Reply with only the digit 0 or 1."
        )
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]}
        ]
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = qwen_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = qwen_model.generate(**inputs, max_new_tokens=5)
        generated = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
        response = qwen_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
        return 0 if "0" in response else 1

    qwen_results = evaluate_prompted_model(
        classify_with_qwen, TEAM_DESCRIPTIONS, frames_dict, gt_dict, clips
    )
    print(f"\nQwen2-VL peak GPU memory: {qwen_mem_mb:.0f} MB\n")
    for name, r in qwen_results.items():
        print(f"  {name}: {r['accuracy']:.1%}  ({r['n_detections']} det, {r['ms_per_crop']:.1f} ms/crop)")
else:
    print("⏭ Skipping Qwen2-VL (RUN_MODEL != 'qwen')")
    print("  To run Qwen2-VL: set RUN_MODEL = 'qwen' in Cell 2, restart runtime, rerun all.")

#### Qwen2-VL — Results and Deliberation

*(Fill in after running — template below. Replace bracketed text with actual numbers.)*

---

The most surprising possible outcome here would be Qwen2-VL *underperforming* SigLIP on
all three clips. That result would be genuinely counterintuitive — Qwen2-VL is a larger,
more capable model that can explicitly reason about jersey descriptions — yet if SigLIP's
embedding approach consistently wins, it means that for this specific task (jersey team
classification from player crops), the embedding strategy is better calibrated than prompted
classification at the 2B scale.

Why might that happen? A few mechanisms:

1. **Prompt sensitivity**: small VLMs can be inconsistent responders. The model may sometimes
   answer "The player is on team 1" (parseable) and sometimes "I cannot determine team
   membership from this image alone" (parsed as 0 by the `"0" in response` heuristic,
   introducing systematic error). Inspect a few raw responses to see.

2. **Resolution bottleneck**: Qwen2-VL 2B processes images at a lower effective resolution
   than the full embedding pipeline. For a tight player crop, the model may be working with
   fewer visual tokens than SigLIP's 14×14 patch grid.

3. **Task mismatch**: Qwen2-VL was trained as a general-purpose instruction follower. SigLIP
   was trained to produce maximally discriminative visual embeddings. For a classification
   task that fundamentally requires distinguishing visual features, a discriminative model may
   have structural advantages over a generative one.

The latency gap between Qwen2-VL and SigLIP is the other key number. If Qwen2-VL is 50–100x
slower *and* not more accurate, the cascade design should treat it as a last-resort stage
for genuinely ambiguous players, not a general-purpose upgrade. The value of Qwen2-VL in the
pipeline likely comes from its reasoning capability in edge cases — particularly jersey number
OCR or reasoning about unusual jersey variants — rather than from raw classification accuracy
on clean player crops.

In [ ]:
# ── Final model comparison table ────────────────────────────────────────────
# florence_results / qwen_results may be None if that model wasn't run
# in this session (incompatible transformers versions — see Cell 2).

all_results = {
    "K-Means": {name: {"accuracy": d["accuracy"], "n_detections": d["n_detections"],
                        "ms_per_crop": None}
                for name, d in kmeans_results.items()},
    "SigLIP":      siglip_results,
    "CLIP":        clip_results,
}
if florence_results is not None:
    all_results["Florence-2"] = florence_results
if qwen_results is not None:
    all_results["Qwen2-VL"] = qwen_results

clips_ordered = ["clip1_easy", "clip2_hard", "clip3_edge"]
header = f"{'Model':<16}" + "".join(f"{c:<22}" for c in clips_ordered)
print(header)
print("-" * len(header))

for model_name, results in all_results.items():
    row = f"{model_name:<16}"
    for c in clips_ordered:
        r   = results[c]
        acc = f"{r['accuracy']:.1%}"
        ms  = f"{r['ms_per_crop']:.0f}ms" if r.get("ms_per_crop") is not None else "—"
        row += f"{acc} ({ms}){'':<8}"
    print(row)

if florence_results is None:
    print("\n(Florence-2 not run — set RUN_MODEL='florence', restart, rerun)")
if qwen_results is None:
    print("\n(Qwen2-VL not run — set RUN_MODEL='qwen', restart, rerun)")

### Summary and Takeaways

*(Fill in after running all models. Template for the key questions to address:)*

---

**Which model should be the primary classifier?**

The answer hinges on clip2_hard. If any model significantly outperforms K-Means there, it
becomes the primary. If all models cluster around 50–60%, clip2 is a genuinely hard failure
mode that requires a different approach — external signals (roster lookup, jersey number OCR)
rather than a better visual model.

**Where does each model add value in a cascade?**

- **K-Means**: near-zero latency, works well when jersey colors are clearly distinct. Good
  first-pass filter — if centroid separation is high and K-Means confidence is high, there's
  no reason to escalate.
- **SigLIP**: richer features, deterministic, still very fast. Natural second stage for cases
  where K-Means is uncertain or centroid separation is low.
- **CLIP**: similar niche to SigLIP; include if SigLIP is unavailable or for cross-validation.
- **Florence-2**: slower but may handle edge cases K-Means and SigLIP can't. Worth routing
  players here when both fast models disagree or produce low confidence.
- **Qwen2-VL**: highest latency, most capable reasoning. Reserve for the hardest cases:
  very low cluster separation, persistent disagreement across earlier stages, or situations
  where jersey number OCR could provide a definitive answer.

**Failure modes that likely persist across all models:**

The 4:1 YOLO detection ratio on clip2_hard (dark jerseys blend into dark arena backgrounds)
affects every approach downstream. Even if a model classifies correctly, it can only classify
players that YOLO successfully detects. Any team with low-contrast jerseys against the
background will be systematically under-represented in the detection stream, regardless of
which classifier runs on top.

**Key findings for `docs/TEAM_CLASSIFICATION_RESEARCH.md`:**

1. Whether embedding models solve the color-similarity failure mode (clip2_hard result)
2. The latency cost of each cascade stage in ms/crop
3. Whether Qwen2-VL's reasoning advantage justifies its latency cost for routine classification
4. Whether response parsing reliability differs between Florence-2 and Qwen2-VL